In [65]:
import csv

TEST_DATA = [
    # Columns: id, data, cliente_id, tipo, valor, descricao, categoria
    [
        "1",
        "2026-01-05",
        "CLI001",
        "credito",
        "3500.00",
        "Salario janeiro",
        "salario",
    ],
    [
        "2",
        "2026-01-12",
        "CLI002",
        "debito",
        "180.50",
        "Supermercado",
        "compra",
    ],
    [
        "3",
        "2026-01-20",
        "CLI001",
        "debito",
        "abc",
        "Erro de sistema",
        "compra",
    ],  # inválida: valor não numérico
    [
        "4",
        "2026-02-03",
        "",
        "debito",
        "200.00",
        "Sem cliente",
        "transferencia",
    ],  # inválida: cliente vazio
    [
        "5",
        "2026-02-14",
        "CLI003",
        "credito",
        "15000.00",
        "Transferencia recebida",
        "transferencia",
    ],  # suspeita
    [
        "6",
        "2026-02-18",
        "CLI002",
        "debito",
        "320.00",
        "Conta de luz",
        "conta",
    ],
    [
        "7",
        "2026-03-01",
        "CLI001",
        "credito",
        "3500.00",
        "Salario marco",
        "salario",
    ],
    [
        "8",
        "2026-03-10",
        "CLI003",
        "debito",
        "99.90",
        "Streaming",
        "assinatura",
    ],
    [
        "9",
        "2026-01-25",
        "CLI002",
        "credito",
        "12000.50",
        "Venda de equipamento",
        "vendas",
    ],  # suspeita
    [
        "10",
        "2026-02-22",
        "CLI001",
        "debito",
        "450.00",
        "Farmacia",
        "saude",
    ],
    [
        "11",
        "2026-02-28",
        "CLI003",
        "debito",
        "89.90",
        "Transporte",
        "transporte",
    ],
    [
        "12",
        "2026-03-15",
        "CLI002",
        "credito",
        "2200.00",
        "Freelance",
        "servicos",
    ],
    [
        "13",
        "2026-03-20",
        "CLI001",
        "debito",
        "780.00",
        "Aluguel",
        "aluguel",
    ],
    [
        "14",
        "2026-01-30",
        "CLI003",
        "credito",
        "950.00",
        "Reembolso",
        "reembolso",
    ],
    [
        "15",
        "2026-03-25",
        "CLI003",
        "credito",
        "3100.00",
        "Projeto extra",
        "servicos",
    ],
    [
        "16",
        "2026-04-02",
        "CLI002",
        "debito",
        "150.00",
        "Assinatura academia",
        "assinatura",
    ],
    [
        "17",
        "2026-04-08",
        "CLI001",
        "credito",
        "3500.00",
        "Salario abril",
        "salario",
    ],
    [
        "18",
        "2026-04-15",
        "CLI002",
        "debito",
        "620.75",
        "Material escolar",
        "educacao",
    ],
    [
        "abc",
        "2026-04-20",
        "CLI003",
        "debito",
        "50.00",
        "ID invalido",
        "compra",
    ],  # inválida: id não numérico
    [
        "20",
        "2026-13-40",
        "CLI001",
        "credito",
        "100.00",
        "Data invalida",
        "compra",
    ],  # inválida: data malformada
    [
        "21",
        "2026-04-22",
        "CLI004",
        "pix",
        "300.00",
        "Tipo invalido",
        "transferencia",
    ],  # inválida: tipo fora do domínio
    [
        "22",
        "2026-04-25",
        "CLI002",
        "debito",
        "-45.00",
        "Valor negativo",
        "compra",
    ],  # inválida: valor <= 0
    [
        "23",
        "2026-04-28",
        "CLI003",
        "credito",
        "0.00",
        "Valor zero",
        "compra",
    ],  # inválida: valor <= 0
]

with open("transacoes.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["id", "data", "cliente_id", "tipo", "valor", "descricao", "categoria"])
    writer.writerows(TEST_DATA)

print(f"transacoes.csv criado com {len(TEST_DATA)} registros")

transacoes.csv criado com 23 registros


In [66]:
import json
from datetime import datetime, date

TRANSACTION_LIMIT = 10000.00
FILENAME = "transacoes.csv"
OUTPUT_FILE = "relatorio.json"


In [67]:
def read_transactions(filename: str) -> list[dict]:
  try:
    with open(filename, mode="r", encoding="utf-8", newline="") as file:
      reader = csv.DictReader(file)
      return list(reader)
  except FileNotFoundError:
    print(f"Erro: aquivo '{filename}' não encontrado.")
    return []

  except Exception as e:
    print(f"Erro ao ler o arquivo: {e}")
    return []

transactions = read_transactions(FILENAME)
print(f"Linhas lidas: {len(transactions)}")

Linhas lidas: 23


In [68]:
# Converte a data AAAA-MM-DD para datetime
def validate_date(date: str) -> datetime | None:
  try:
    return datetime.strptime(date, "%Y-%m-%d")
  except (ValueError, TypeError):
    return None


# Validar valor da transação
def validate_value(value: str) -> float | None:
  try:
    float_value = float(value)

    return float_value if float_value > 0 else None
  except (ValueError, TypeError):
    return None


# Validar transação
def validate_transaction(line: dict) -> dict | None:
  try:
    id = (line.get("id") or "").strip()

    if not id or not id.isdigit():
      return None

    client_id = (line.get("cliente_id") or "").strip()

    if not client_id:
      return None

    date = validate_date(line.get("data", "").strip())

    if not date:
      return None

    transaction_type = (line.get("tipo") or "").strip().lower()
    if transaction_type not in ("credito", "debito"):
      return None


    value = validate_value((line.get("valor") or "").strip())

    if value is None:
      return None

    return {
        "id": int(id),
        "date": date,
        "client_id": client_id,
        "type": transaction_type,
        "value": value,
        "description": (line.get("descricao") or "").strip(),
        "category": (line.get("categoria") or "").strip(),
    }

  except (KeyError, ValueError):
    print(f"Erro ao validar transação: {e}")
    return None


valid = validate_transaction({"id": "1", "data": "2026-01-05", "cliente_id": "CLI001",
                            "tipo": "credito", "valor": "3500.00", "descricao": "x", "categoria": "y"})
invalid = validate_transaction({"id": "3", "data": "2026-01-20", "cliente_id": "CLI001",
                               "tipo": "debito", "valor": "abc", "descricao": "x", "categoria": "y"})
print("Válida ->", valid)
print("\nInválida ->", invalid)

Válida -> {'id': 1, 'date': datetime.datetime(2026, 1, 5, 0, 0), 'client_id': 'CLI001', 'type': 'credito', 'value': 3500.0, 'description': 'x', 'category': 'y'}

Inválida -> None


In [69]:
def generate_report(validated_transactions: list[dict]) -> dict:
  report: dict[str, dict] = {}

  for t in validated_transactions:
    month = t["date"].strftime("%Y-%m")
    if month not in report:
      report[month] = {
          "quantidade": 0,
          "total_credito": 0.0,
          "total_debito": 0.0,
          "saldo": 0.0,
          "media": 0.0,
          "maior_valor": 0.0,
          "menor_valor": float("inf")
      }

    r = report[month]
    r["quantidade"] += 1
    if t["type"] == "credito":
      r["total_credito"] += t["value"]
      r["saldo"] += t["value"]
    else:
      r["total_debito"] += t["value"]
      r["saldo"] -= t["value"]

    r["maior_valor"] = max(r["maior_valor"], t["value"])
    r["menor_valor"] = min(r["menor_valor"], t["value"])

  for r in report.values():
    r["media"] = (r["total_credito"] + r["total_debito"]) / r["quantidade"]
    if r["menor_valor"] == float("inf"):
      r["menor_valor"] = 0.0

  return report


def calculate_period(validated_transactions: list[dict]) -> tuple[datetime, datetime, int]:
  min_date = min(t["date"] for t in validated_transactions)
  max_date = max(t["date"] for t in validated_transactions)
  period_days = (max_date - min_date).days

  return min_date, max_date, period_days


valid_transactions = [r for r in (validate_transaction(l) for l in transactions) if r is not None]
transactions_list = generate_report(valid_transactions)
for month, r in sorted(transactions_list.items()):
    print(month, "->", r)



2026-01 -> {'quantidade': 4, 'total_credito': 16450.5, 'total_debito': 180.5, 'saldo': 16270.0, 'media': 4157.75, 'maior_valor': 12000.5, 'menor_valor': 180.5}
2026-02 -> {'quantidade': 4, 'total_credito': 15000.0, 'total_debito': 859.9, 'saldo': 14140.1, 'media': 3964.975, 'maior_valor': 15000.0, 'menor_valor': 89.9}
2026-03 -> {'quantidade': 5, 'total_credito': 8800.0, 'total_debito': 879.9, 'saldo': 7920.1, 'media': 1935.98, 'maior_valor': 3500.0, 'menor_valor': 99.9}
2026-04 -> {'quantidade': 3, 'total_credito': 3500.0, 'total_debito': 770.75, 'saldo': 2729.25, 'media': 1423.5833333333333, 'maior_valor': 3500.0, 'menor_valor': 150.0}


In [70]:
def generate_report_json(validated_transactions: list[dict], invalid_transactions: int, report: dict, warning_transactions: list[dict], min_date: datetime, max_date: datetime, days: int) -> dict:

    return {
        "gerado_em": date.today().isoformat(),
        "total_transacoes_validas": len(validated_transactions),
        "total_transacoes_invalidas": invalid_transactions,
        "periodo": {
            "mais_antiga": min_date.strftime("%Y-%m-%d"),
            "mais_recente": max_date.strftime("%Y-%m-%d"),
            "dias_entre": days,
        },
        "resumo_mensal": {
            month: {
                "quantidade": r["quantidade"],
                "total_credito": round(r["total_credito"], 2),
                "total_debito": round(r["total_debito"], 2),
                "saldo": round(r["saldo"], 2),
                "media": round(r["media"], 2),
                "maior_valor": round(r["maior_valor"], 2),
                "menor_valor": round(r["menor_valor"], 2),
            }
            for month, r in sorted(report.items())
        },
        "transacoes_suspeitas": [
            {
                "id": s["id"],
                "cliente_id": s["client_id"],
                "data": s["date"].strftime("%Y-%m-%d"),
                "valor": s["value"],
            }
            for s in warning_transactions
        ],
    }

def save_json(dados: dict, filename: str = OUTPUT_FILE) -> None:
    try:
        with open(filename, mode="w", encoding="utf-8") as f:
            json.dump(dados, f, ensure_ascii=False, indent=2)
        print(f"Relatório salvo em '{filename}'.")
    except OSError as error:
        print(f"Erro ao salvar o JSON: {error}")

In [71]:
def format_money(value: float) -> str:
  return f"R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def show_report(report: dict, warning_transactions: list[dict], valid_transactions_count: int, invalid_transactions_count: int, min_date: datetime, max_date: datetime) -> None:
  print("=" * 45)
  print("       RELATÓRIO FINANCEIRO — CLEARBANK")
  print("=" * 45)
  print(f"Período analisado: {min_date:%Y-%m-%d} → {max_date:%Y-%m-%d}")
  print(f"Transações válidas:   {valid_transactions_count}")
  print(f"Transações inválidas: {invalid_transactions_count}")

  print("\n===== RELATÓRIO MENSAL =====")
  for month in sorted(report):
    r = report[month]
    print(f"\nMês: {month}")
    print(f"  Transações: {r['quantidade']}")
    print(f"  Total crédito: {format_money(r['total_credito'])}")
    print(f"  Total débito:  {format_money(r['total_debito'])}")
    print(f"  Saldo:         {format_money(r['saldo'])}")
    print(f"  Média:         {format_money(r['media'])}")
    print(f"  Maior valor:   {format_money(r['maior_valor'])}")
    print(f"  Menor valor:   {format_money(r['menor_valor'])}")

  print("\n===== TRANSAÇÕES SUSPEITAS =====")
  if not warning_transactions:
        print("Nenhuma transação suspeita encontrada.")
  else:
    for s in warning_transactions:
      print(f"ID: {s['id']} | Cliente: {s['client_id']} | "
              f"Data: {s['date']:%Y-%m-%d} | Valor: {format_money(s['value'])}")

In [72]:
def main() -> None:
  transactions = read_transactions(FILENAME)
  total_transactions = len(transactions)

  valid_transactions: list[dict] = []
  warning_transactions: list[dict] = []

  for line in transactions:
    transaction = validate_transaction(line)
    if transaction is None:
      continue

    valid_transactions.append(transaction)

    if transaction["value"] > TRANSACTION_LIMIT:
      warning_transactions.append(transaction)

  invalid_transactions = total_transactions - len(valid_transactions)

  print(f"Total de linhas lidas: {total_transactions}")
  print(f"Linhas válidas: {len(valid_transactions)}")
  print(f"Linhas inválidas: {invalid_transactions}")

  if not valid_transactions:
    print("Nenhuma transação válida encontrada.")
    return

  report = generate_report(valid_transactions)
  min_date, max_date, days = calculate_period(valid_transactions)

  show_report(report, warning_transactions, len(valid_transactions), invalid_transactions, min_date, max_date)

  print(f"\nDias entre a transação mais antiga e a mais recente: {days}")

  report_json = generate_report_json(valid_transactions, invalid_transactions, report, warning_transactions, min_date, max_date, days)

  save_json(report_json)


main()

Total de linhas lidas: 23
Linhas válidas: 16
Linhas inválidas: 7
       RELATÓRIO FINANCEIRO — CLEARBANK
Período analisado: 2026-01-05 → 2026-04-15
Transações válidas:   16
Transações inválidas: 7

===== RELATÓRIO MENSAL =====

Mês: 2026-01
  Transações: 4
  Total crédito: R$ 16.450,50
  Total débito:  R$ 180,50
  Saldo:         R$ 16.270,00
  Média:         R$ 4.157,75
  Maior valor:   R$ 12.000,50
  Menor valor:   R$ 180,50

Mês: 2026-02
  Transações: 4
  Total crédito: R$ 15.000,00
  Total débito:  R$ 859,90
  Saldo:         R$ 14.140,10
  Média:         R$ 3.964,97
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 89,90

Mês: 2026-03
  Transações: 5
  Total crédito: R$ 8.800,00
  Total débito:  R$ 879,90
  Saldo:         R$ 7.920,10
  Média:         R$ 1.935,98
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 99,90

Mês: 2026-04
  Transações: 3
  Total crédito: R$ 3.500,00
  Total débito:  R$ 770,75
  Saldo:         R$ 2.729,25
  Média:         R$ 1.423,58
  Maior valor:   R$ 3.500